<a href="https://colab.research.google.com/github/smagadi/AIML/blob/master/Trading/StockPredictionML_ver1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance pandas ta scikit-learn matplotlib seaborn


In [3]:
import yfinance as yf
import ta
import pandas as pd # Import pandas

def get_stock_data(ticker, period='2y', interval='1d'):
    df = yf.download(ticker, period=period, interval=interval)
    df.dropna(inplace=True)
    return df

df = get_stock_data('NVDA')  # Replace 'NVDA' with 'AMZN' for Amazon
print(df.head())



YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           NVDA       NVDA       NVDA       NVDA       NVDA
Date                                                             
2023-06-14  42.970554  42.973551  40.527055  40.798891  740465000
2023-06-15  42.626770  43.262380  42.121081  42.575803  568622000
2023-06-16  42.665745  43.694111  42.634762  43.423278  655709000
2023-06-20  43.781055  43.962947  42.647754  42.971556  451153000
2023-06-21  43.018524  43.588177  42.054121  43.474245  551603000


In [13]:
def add_Tech_indi(df):
    # Ensure 'Close' is a pandas Series
    if isinstance(df['Close'], pd.DataFrame):
        close = df['Close'].iloc[:, 0]
    else:
        close = df['Close']

    # Use the Series in indicators
    df['EMA'] = ta.trend.ema_indicator(close=close, window=20)
    df['RSI'] = ta.momentum.rsi(close=close, window=14)
    macd = ta.trend.MACD(close=close)
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(close=close)
    df['BB_upper'] = bb.bollinger_hband()
    df['BB_lower'] = bb.bollinger_lband()




    df.dropna(inplace=True)
    return df


In [14]:
df = get_stock_data('NVDA')  # Replace 'NVDA' with 'AMZN' for Amazon

print("Close shape:", df['Close'].shape)

[*********************100%***********************]  1 of 1 completed

Close shape: (502, 1)


In [15]:
df = add_Tech_indi(df)

In [16]:
#Model preparaation
#Lag the target to avoid lookahead bias:
print("Shape before adding Target:", df.shape)
df['Target'] = df['Close'].shift(-1)
print("Shape after adding Target:", df.shape)
df.dropna(inplace=True)
print("Shape after dropping NaNs post-Target:", df.shape)


Shape before adding Target: (469, 11)
Shape after adding Target: (469, 12)
Shape after dropping NaNs post-Target: (468, 12)


In [17]:
#Select features and split data:
print("Shape of df before feature selection:", df.shape)
features = ['EMA', 'RSI', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'Volume']
X = df[features]
y = df['Target']
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)


Shape of df before feature selection: (468, 12)
Shape of X: (468, 7)
Shape of y: (468,)


In [18]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

In [19]:
#5. Model Training & Validation
#Random Forest with Time Series Cross-Validation:

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

mae_scores = []
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

print(f"Average MAE: {sum(mae_scores)/len(mae_scores):.4f}")


Average MAE: 11.9880


In [21]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# The previous code blocks for data loading, indicator adding, and model training are assumed to have run successfully.
# Specifically, the DataFrame 'df', the features list 'features',
# the feature DataFrame 'X', the target Series 'y', and the trained 'model'
# from the TimeSeriesSplit cross-validation loop are expected to be available.

last_X = X.iloc[-1:].copy()
predictions = []
model.fit(X, y)  # Retrain on all data for final prediction

for _ in range(10):
    pred = model.predict(last_X)[0]
    predictions.append(pred)

    # Update last_X with predicted value (simulate next day)
    # NOTE: These are simplified updates for demonstration.
    # In a production environment, you would need a more sophisticated
    # method to calculate future technical indicators based on new predicted data.
    last_X = last_X.copy()

    # Correcting column names to match the features list and df columns
    last_X['EMA'] = (last_X['EMA'] * 9 + pred) / 10  # Simple EMA update, assume window 10 (approximation)
    # The following lines are placeholders. Updating these indicators correctly
    # requires the previous price data points, which are not available in `last_X`.
    # For a true forecast, you would need to regenerate the indicators based on
    # the predicted price sequence.
    last_X['RSI'] = last_X['RSI'] # RSI static for demo, update with real calc in prod
    last_X['MACD'] = last_X['MACD'] # Similar for MACD
    last_X['MACD_signal'] = last_X['MACD_signal']
    last_X['BB_upper'] = last_X['BB_upper']
    last_X['BB_lower'] = last_X['BB_lower']
    last_X['Volume'] = last_X['Volume'] # Volume is also challenging to predict/simulate accurately

print("Next 10 predicted closing prices:")
for i, p in enumerate(predictions, 1):
    print(f"Day {i}: ${p:.2f}")

Next 10 predicted closing prices:
Day 1: $143.04
Day 2: $143.26
Day 3: $143.47
Day 4: $143.35
Day 5: $143.28
Day 6: $142.86
Day 7: $141.86
Day 8: $141.84
Day 9: $141.84
Day 10: $141.84
